# Volume 2. FSM And PFA Utilities

Question: how do the package-level automata helpers expose discrete computation over assembly states?

This is the state-machine walk. You will first build a deterministic toggle, then a probabilistic branch. The names are readable (`even`, `odd`, `left`, `right`), but the helpers still keep assembly snapshots underneath for inspection.

In [ ]:
# Counter lets us summarize repeated probabilistic samples.
from collections import Counter

# Brain provides the assembly substrate underneath the automata helpers.
from neural_assemblies.core.brain import Brain

# FSMNetwork and PFANetwork expose readable state-machine APIs.
from neural_assemblies.assembly_calculus import FSMNetwork, PFANetwork, overlap

First build a deterministic parity machine. The state toggles each time it reads `1`. This is the simplest useful computation: the machine must remember whether it has seen an even or odd number of symbols so far.

In [ ]:
# Use one brain for the toy automata examples in this notebook.
brain = Brain(p=0.05, save_winners=True, seed=42, engine="numpy_sparse")

# This FSM tracks parity: every "1" toggles even <-> odd.
fsm = FSMNetwork(
    brain,
    states=["even", "odd"],
    symbols=["1"],
    transitions=[("even", "1", "odd"), ("odd", "1", "even")],
    initial_state="even",
    n=4_000,
    k=60,
    beta=0.05,
    rounds=6,
)

# The trajectory is the readable state trace after consuming four symbols.
print("initial:", fsm.current_state)
print("trajectory:", fsm.run(["1", "1", "1", "1"]))

The state lexicon is inspectable. The two state assemblies should be much less similar than an assembly compared with itself.

In [ ]:
# State names map to assembly snapshots in the state lexicon.
even = fsm.state_lexicon["even"]
odd = fsm.state_lexicon["odd"]

# Self-overlap is 1.0; even-vs-odd should be much lower.
print("overlap(even, even):", overlap(even, even))
print("overlap(even, odd):", f"{overlap(even, odd):.3f}")

Now build a probabilistic automaton. Reset before each sample when you want independent draws from the same initial state.

In [ ]:
# This PFA branches from start to left or right with equal probability.
pfa = PFANetwork(
    brain,
    states=["start", "left", "right"],
    symbols=["go"],
    transitions=[("start", "go", "left", 0.5), ("start", "go", "right", 0.5)],
    initial_state="start",
    n=4_000,
    k=60,
    beta=0.05,
    rounds=6,
)

# Reset before each sample so every draw starts from the same state.
counts = Counter()
for seed in range(12):
    pfa.reset()
    # The seed makes the stochastic choice reproducible for the notebook.
    counts[pfa.step("go", seed=seed)] += 1

# With only 12 samples, expect rough stochastic behavior, not exact 50/50 counts.
print(dict(counts))

What to notice: the API gives you readable state names and trajectories. The underlying implementation still uses assemblies internally, so you can inspect overlaps when debugging.

Try next: add a second symbol, or change the PFA probabilities from `0.5/0.5` to `0.8/0.2`. Keep the reset before each sample if you want independent draws from the same start state.